# AgriNexus AI — Research-Grade Notebook 06: Soil Organic Carbon Analysis

**Task**: Soil Organic Carbon (`OC`, g/kg) Regression on LUCAS European Topsoil Dataset
**Primary Dataset**: `LUCAS_Topsoil_2015_20200323.csv` (21,859 EU Soil Survey Observations)
**Scientific Focus**: Target Terminology Correction (Soil Organic Carbon OC, not general 'soil health'), Spatial Group-Aware Splitting (`NUTS_2` Region GroupShuffleSplit), Multi-Model Regressor Suite Benchmarking, Empirical Residual-Based Prediction Intervals (Uncertainty & Nominal Coverage Report), European Geographic Limitations, and Artifact Reload Verification.

In [1]:
# Section 1: Environment, Dependencies & Deterministic Seed Setup
import os
import sys
import math
import time
import json
import random
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error

warnings.filterwarnings('ignore')

# Deterministic Seed Setup
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

DATA_DIR = Path('d:/PROJECTS/AGRINEXUS-AI/data/raw/soil_analysis')
if not DATA_DIR.exists():
    DATA_DIR = Path('../data/raw/soil_analysis')

MODELS_DIR = Path('d:/PROJECTS/AGRINEXUS-AI/Notebook/models')
if not MODELS_DIR.exists():
    MODELS_DIR = Path('models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Environment Ready | Seed: {SEED}")
print(f"Data Path: {DATA_DIR.resolve()}")
print(f"Models Directory: {MODELS_DIR.resolve()}")

Environment Ready | Seed: 42
Data Path: D:\PROJECTS\AGRINEXUS-AI\data\raw\soil_analysis
Models Directory: D:\PROJECTS\AGRINEXUS-AI\Notebook\models


## 2. Problem Statement & Spatial Leakage Audit
Soil Organic Carbon ($OC$, in g/kg) is a key indicator of soil organic matter and fertility. Topsoil samples exhibit spatial autocorrelation across geographical regions.

> [!IMPORTANT]
> **Scientific Corrections**:
> 1. **Target**: Strictly defined as **Soil Organic Carbon (OC)**, not general "soil health".
> 2. **Spatial Split**: We use `GroupShuffleSplit` on `NUTS_2` territorial regions for holdout partitions to ensure zero geographic region overlap between train, val, and test sets. Cross-validation within training partition uses `GroupKFold`.
> 3. **Uncertainty**: Residual quantile bounds are designated **Empirical residual-based prediction intervals**, not guaranteed conformal prediction.

In [2]:
# Section 3: Data Ingestion & Spatial Preprocessing
csv_path = DATA_DIR / "LUCAS_Topsoil_2015_20200323.csv"
if not csv_path.exists():
    csv_path = DATA_DIR / "lucas_2015_topsoil" / "LUCAS_Topsoil_2015_20200323.csv"

df_raw = pd.read_csv(csv_path)
print(f"Raw LUCAS Dataset Loaded: {len(df_raw):,} rows, {len(df_raw.columns)} columns")

target_col = 'OC'
spatial_group_col = 'NUTS_2'

df_clean = df_raw.dropna(subset=[target_col, spatial_group_col]).copy()
print(f"Clean Dataset Size (valid target & NUTS_2): {len(df_clean):,} rows")

candidate_features = ['Coarse', 'Clay', 'Sand', 'Silt', 'pH(CaCl2)', 'pH(H2O)', 'EC', 'P', 'N', 'K', 'Elevation', 'LC1', 'LU1']
feature_cols = [c for c in candidate_features if c in df_clean.columns]
num_cols = list(df_clean[feature_cols].select_dtypes(include=[np.number]).columns)
cat_cols = list(df_clean[feature_cols].select_dtypes(include=['object']).columns)

print(f"Target Column: '{target_col}' | Range: min={df_clean[target_col].min():.2f}, max={df_clean[target_col].max():.2f} g/kg")
print(f"Spatial Region Groups (NUTS_2): {df_clean[spatial_group_col].nunique()} unique regions")
print(f"Numerical Features ({len(num_cols)}): {num_cols}")
print(f"Categorical Features ({len(cat_cols)}): {cat_cols}")

Raw LUCAS Dataset Loaded: 21,859 rows, 24 columns
Clean Dataset Size (valid target & NUTS_2): 21,859 rows


Target Column: 'OC' | Range: min=0.10, max=560.20 g/kg
Spatial Region Groups (NUTS_2): 259 unique regions
Numerical Features (11): ['Coarse', 'Clay', 'Sand', 'Silt', 'pH(CaCl2)', 'pH(H2O)', 'EC', 'P', 'N', 'K', 'Elevation']
Categorical Features (2): ['LC1', 'LU1']


In [3]:
# Section 4: Spatial Group-Aware Partitioning (GroupShuffleSplit on NUTS_2)
groups = df_clean[spatial_group_col].values

# GroupShuffleSplit: 70% Train, 15% Val, 15% Test based on NUTS_2 regions
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(gss1.split(df_clean, groups=groups))

df_train = df_clean.iloc[train_idx].reset_index(drop=True)
df_temp = df_clean.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp[spatial_group_col].values))

df_val = df_temp.iloc[val_idx].reset_index(drop=True)
df_test = df_temp.iloc[test_idx].reset_index(drop=True)

train_regions = set(df_train[spatial_group_col].unique())
val_regions = set(df_val[spatial_group_col].unique())
test_regions = set(df_test[spatial_group_col].unique())

print(f"Spatial Partitioning Statistics:")
print(f"  - Train: {len(df_train):,} samples across {len(train_regions)} NUTS_2 regions")
print(f"  - Val:   {len(df_val):,} samples across {len(val_regions)} NUTS_2 regions")
print(f"  - Test:  {len(df_test):,} samples across {len(test_regions)} NUTS_2 regions")

# Spatial Leakage Verification Assertions
assert len(train_regions.intersection(val_regions)) == 0, "Spatial leakage detected between Train and Val!"
assert len(train_regions.intersection(test_regions)) == 0, "Spatial leakage detected between Train and Test!"
print("Spatial Leakage Audit Passed: ZERO spatial region overlap across splits.")

X_train, y_train = df_train[feature_cols], df_train[target_col].values
X_val, y_val = df_val[feature_cols], df_val[target_col].values
X_test, y_test = df_test[feature_cols], df_test[target_col].values

Spatial Partitioning Statistics:
  - Train: 15,380 samples across 181 NUTS_2 regions
  - Val:   3,868 samples across 39 NUTS_2 regions
  - Test:  2,611 samples across 39 NUTS_2 regions
Spatial Leakage Audit Passed: ZERO spatial region overlap across splits.


In [4]:
# Section 5: Preprocessing ColumnTransformer & Regressor Benchmarking
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc = preprocessor.transform(X_val)
X_test_proc = preprocessor.transform(X_test)

regressors = {
    'Dummy Baseline (Mean)': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Ridge Baseline': Ridge(alpha=10.0, random_state=SEED),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=SEED, n_jobs=-1),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=100, random_state=SEED),
    'LightGBM Regressor': lgb.LGBMRegressor(n_estimators=100, max_depth=6, learning_rate=0.05, random_state=SEED, n_jobs=-1, verbose=-1),
    'XGBoost Regressor': xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.05, random_state=SEED, n_jobs=-1)
}

benchmark_results = []
best_val_r2 = -float('inf')
best_model_name = None
best_model = None

print("Benchmarking Candidate Regressors on Validation Partition...")
for name, reg in regressors.items():
    reg.fit(X_train_proc, y_train)
    val_preds = reg.predict(X_val_proc)
    
    mae = mean_absolute_error(y_val, val_preds)
    rmse = math.sqrt(mean_squared_error(y_val, val_preds))
    r2 = r2_score(y_val, val_preds)
    
    benchmark_results.append({
        'Model': name,
        'Val MAE': mae,
        'Val RMSE': rmse,
        'Val R2': r2
    })
    print(f"  {name:<24} | Val MAE: {mae:7.4f} g/kg | Val RMSE: {rmse:7.4f} | Val R2: {r2:6.4f}")
    
    if r2 > best_val_r2:
        best_val_r2 = r2
        best_model_name = name
        best_model = reg

df_bench = pd.DataFrame(benchmark_results)
print(f"\nCHAMPION REGRESSOR SELECTED (via Validation R2): {best_model_name} (Val R2 = {best_val_r2:.4f})")

Benchmarking Candidate Regressors on Validation Partition...
  Dummy Baseline (Mean)    | Val MAE: 48.6874 g/kg | Val RMSE: 104.4875 | Val R2: -0.0396


  Linear Regression        | Val MAE: 17.5429 g/kg | Val RMSE: 35.4029 | Val R2: 0.8806


  Ridge Baseline           | Val MAE: 17.4691 g/kg | Val RMSE: 35.4608 | Val R2: 0.8803


  Random Forest            | Val MAE:  9.4262 g/kg | Val RMSE: 21.5154 | Val R2: 0.9559


  HistGradientBoosting     | Val MAE:  9.2178 g/kg | Val RMSE: 20.5981 | Val R2: 0.9596


  LightGBM Regressor       | Val MAE:  9.1998 g/kg | Val RMSE: 20.9013 | Val R2: 0.9584


  XGBoost Regressor        | Val MAE:  9.1444 g/kg | Val RMSE: 20.8667 | Val R2: 0.9585

CHAMPION REGRESSOR SELECTED (via Validation R2): HistGradientBoosting (Val R2 = 0.9596)


In [5]:
# Section 6: Out-of-Region Test Set Evaluation & Empirical Prediction Intervals
test_preds = best_model.predict(X_test_proc)

test_mae = mean_absolute_error(y_test, test_preds)
test_rmse = math.sqrt(mean_squared_error(y_test, test_preds))
test_r2 = r2_score(y_test, test_preds)
test_medae = median_absolute_error(y_test, test_preds)

print(f"Final Out-of-Region Test Performance for Champion ({best_model_name}):")
print(f"  - Test MAE:                  {test_mae:.4f} g/kg")
print(f"  - Test RMSE:                 {test_rmse:.4f} g/kg")
print(f"  - Test R2 Score:             {test_r2:.4f}")
print(f"  - Test Median Absolute Error: {test_medae:.4f} g/kg")

# Empirical Residual-Based Prediction Interval Bounds (95th Percentile Validation Residual)
val_preds = best_model.predict(X_val_proc)
val_residuals = np.abs(y_val - val_preds)
q95_residual = float(np.percentile(val_residuals, 95))

# Compute empirical test coverage
lower_bounds = test_preds - q95_residual
upper_bounds = test_preds + q95_residual
in_interval = (y_test >= lower_bounds) & (y_test <= upper_bounds)
observed_coverage = np.mean(in_interval) * 100.0

print(f"\nEmpirical Residual-Based Prediction Interval Audit:")
print(f"  - 95th Percentile Validation Residual Bound (q95): {q95_residual:.4f} g/kg")
print(f"  - Nominal Coverage Goal: 95.00%")
print(f"  - Observed Test Set Coverage: {observed_coverage:.2f}%")
print(f"  - Average Interval Width: {2 * q95_residual:.4f} g/kg")

Final Out-of-Region Test Performance for Champion (HistGradientBoosting):


  - Test MAE:                  6.5481 g/kg
  - Test RMSE:                 13.4570 g/kg
  - Test R2 Score:             0.9614
  - Test Median Absolute Error: 2.8644 g/kg

Empirical Residual-Based Prediction Interval Audit:
  - 95th Percentile Validation Residual Bound (q95): 40.4529 g/kg
  - Nominal Coverage Goal: 95.00%
  - Observed Test Set Coverage: 97.43%
  - Average Interval Width: 80.9058 g/kg


## 7. Geographic Scope & Domain Limitations
The model is trained strictly on topsoil samples from the European Union (LUCAS 2015 Survey).

> [!WARNING]
> **Geographic Limitation**: This model cannot be deployed directly to tropical Indian agro-ecological zones without local domain re-calibration, as soil organic matter dynamics differ substantially under tropical monsoon regimes.

In [6]:
# Section 8: Model Artifact Serialization & Reload Verification
artifact_filename = "soil_analysis.pkl"
artifact_path = MODELS_DIR / artifact_filename

export_package = {
    'preprocessor': preprocessor,
    'model': best_model,
    'best_model_name': best_model_name,
    'feature_cols': feature_cols,
    'num_cols': num_cols,
    'cat_cols': cat_cols,
    'q95_residual': q95_residual,
    'target_col': target_col,
    'metadata': {
        'dataset_name': 'LUCAS 2015 EU Topsoil Dataset',
        'target_variable': 'Soil Organic Carbon (OC)',
        'train_samples': len(X_train),
        'val_samples': len(X_val),
        'test_samples': len(X_test),
        'test_mae': float(test_mae),
        'test_rmse': float(test_rmse),
        'test_r2': float(test_r2),
        'observed_coverage_pct': float(observed_coverage),
        'random_seed': SEED,
        'saved_at': time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())
    }
}

with open(artifact_path, 'wb') as f:
    pickle.dump(export_package, f)

artifact_size_mb = artifact_path.stat().st_size / (1024 * 1024)
print("Artifact Overwritten Successfully!")
print(f"  - Path: {artifact_path.resolve()}")
print(f"  - Size: {artifact_size_mb:.2f} MB")

# Reload Verification Check
with open(artifact_path, 'rb') as f:
    reloaded_dict = pickle.load(f)

reloaded_prep = reloaded_dict['preprocessor']
reloaded_model = reloaded_dict['model']

X_sample = X_test.iloc[:10]
y_orig_sample = best_model.predict(X_test_proc[:10])
y_reload_sample = reloaded_model.predict(reloaded_prep.transform(X_sample))

is_deterministic = np.allclose(y_orig_sample, y_reload_sample, atol=1e-5)
print(f"\nArtifact Reload Verification Check: Predictions Match 100%: {is_deterministic}")
assert is_deterministic, "CRITICAL FAILURE: Reloaded soil model predictions do not match!"
print("QUALITY GATE PASSED: Soil organic carbon artifact reloaded cleanly.")

Artifact Overwritten Successfully!
  - Path: D:\PROJECTS\AGRINEXUS-AI\Notebook\models\soil_analysis.pkl
  - Size: 0.26 MB



Artifact Reload Verification Check: Predictions Match 100%: True
QUALITY GATE PASSED: Soil organic carbon artifact reloaded cleanly.


In [7]:
# Section 9: Final Scientific Audit Table & Conclusions
readiness = "PASS" if (test_r2 >= 0.25 and is_deterministic) else "CONDITIONAL"

final_audit_summary = [
    {"Metric / Aspect": "Dataset", "Audit Value": "LUCAS 2015 EU Topsoil Survey Dataset"},
    {"Metric / Aspect": "Dataset Size", "Audit Value": f"{len(df_clean):,} valid observations ({len(X_train):,} train, {len(X_val):,} val, {len(X_test):,} test)"},
    {"Metric / Aspect": "Target Variable", "Audit Value": "OC (Soil Organic Carbon)"},
    {"Metric / Aspect": "Target Unit", "Audit Value": "g/kg (grams per kilogram soil)"},
    {"Metric / Aspect": "Features", "Audit Value": f"{len(feature_cols)} soil physical/chemical & land cover features"},
    {"Metric / Aspect": "Split Strategy", "Audit Value": "Spatial Region GroupShuffleSplit (NUTS_2 territorial groups)"},
    {"Metric / Aspect": "Leakage Audit", "Audit Value": "PASS (Zero spatial NUTS_2 region overlap across splits)"},
    {"Metric / Aspect": "Baseline Model", "Audit Value": "DummyRegressor (Mean Target) & Ridge Baseline"},
    {"Metric / Aspect": "Candidate Models", "Audit Value": "Dummy, Linear, Ridge, Random Forest, HistGB, LightGBM, XGBoost"},
    {"Metric / Aspect": "Champion Model", "Audit Value": f"{best_model_name} (Selected via Validation R2)"},
    {"Metric / Aspect": "Validation Metric", "Audit Value": f"Val R2 = {best_val_r2:.4f}"},
    {"Metric / Aspect": "Held-Out Test Metric", "Audit Value": f"Test R2 = {test_r2:.4f}, MAE = {test_mae:.4f} g/kg, RMSE = {test_rmse:.4f}"},
    {"Metric / Aspect": "Uncertainty Interval", "Audit Value": f"Empirical 95% residual bound (q95 = {q95_residual:.4f} g/kg, Test Coverage = {observed_coverage:.2f}%)"},
    {"Metric / Aspect": "Geographic Scope", "Audit Value": "Validated on European soils (LUCAS); requires domain adaptation for non-EU deployment"},
    {"Metric / Aspect": "Artifact Reload Result", "Audit Value": "PASS (Exact pipeline state prediction match)"},
    {"Metric / Aspect": "Readiness Status", "Audit Value": readiness}
]

df_audit_summary = pd.DataFrame(final_audit_summary)
print("="*70)
print("FINAL MODEL AUDIT REPORT — SOIL ORGANIC CARBON ANALYSIS")
print("="*70)
print(df_audit_summary.to_string(index=False))
print("="*70)

FINAL MODEL AUDIT REPORT — SOIL ORGANIC CARBON ANALYSIS
       Metric / Aspect                                                                           Audit Value
               Dataset                                                  LUCAS 2015 EU Topsoil Survey Dataset
          Dataset Size                       21,859 valid observations (15,380 train, 3,868 val, 2,611 test)
       Target Variable                                                              OC (Soil Organic Carbon)
           Target Unit                                                        g/kg (grams per kilogram soil)
              Features                                       13 soil physical/chemical & land cover features
        Split Strategy                          Spatial Region GroupShuffleSplit (NUTS_2 territorial groups)
         Leakage Audit                               PASS (Zero spatial NUTS_2 region overlap across splits)
        Baseline Model                                         DummyRegr